In [1]:
# Importing libraries
import os
import re
import numpy as np
import pandas as pd

from qsarmodelingpy.external_validation import ExternalValidation
from qsarmodelingpy.cross_validation_class import CrossValidation
from qsarmodelingpy.kennardstonealgorithm import kennardstonealgorithm
import qsarmodelingpy.lj_cut as lj
from qsarmodelingpy.calculate_parameters import calcR2

from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource
from bokeh.io import export_png

In [2]:
# Open configuration file in order to look for the matrices and the parameters to run
# external validation
dfConf = pd.read_csv(
    "/home/helitonmrf/Documents/TEMP/qsarm_tests/graficos/confExtVal.csv", header=None
)
directory = dfConf[1][0]
Xfile = dfConf[1][1]
yfile = dfConf[1][2]
nLV = None if dfConf.isnull()[1][4] else int(dfConf[1][4])
out_directory = dfConf[1][5]
ext_val_file = dfConf[1][6]
cv_file = dfConf[1][7]
Xtrain_file = dfConf[1][8]
ytrain_file = dfConf[1][9]
Xtest_file = dfConf[1][10]
ytest_file = dfConf[1][11]
autoscale = dfConf[1][12].upper() == "YES"
y = pd.read_csv(os.path.join(directory, yfile), sep=";", header=None).values
dfX = pd.read_csv(os.path.join(directory, Xfile), sep=";", index_col=0)
dfX = lj.transform(dfX) if dfConf[1][13].upper() == "YES" else dfX
X = dfX.values
type_ext_val = int(dfConf[1][14])
if type_ext_val == 1:  # manual selection
    test_set = dfConf[1][3]
    test = [int(i) - 1 for i in test_set.split(",")]
    train = [j for j in range(len(y)) if j not in test]
elif type_ext_val == 2:  # Kennard-Stone
    size_test_set = int(dfConf[1][3])
    train, test = kennardstonealgorithm(
        dfX, len(dfX) - size_test_set
    )  # parameter is the size of training set
else:  # Random selection
    pass
ext = ExternalValidation(X, y, nLV)
ext.extVal(train, test, nLV)
ext.saveExtVal(train, test, out_directory + "/" + ext_val_file)
cv = CrossValidation(X[train, :], y[train], nLVMax=nLV, scale=True)
cv.saveParameters(os.path.join(out_directory, cv_file))
dfXtrain = dfX.loc[dfX.index[train], dfX.columns]
dfXtrain.to_csv(os.path.join(out_directory, Xtrain_file), sep=";")
dfytrain = pd.DataFrame(y[train])
dfytrain.to_csv(os.path.join(out_directory, ytrain_file), header=False)
dfXtest = dfX.loc[dfX.index[test], dfX.columns]
dfXtest.to_csv(os.path.join(out_directory, Xtest_file), sep=";")
dfytest = pd.DataFrame(y[test])
dfytest.to_csv(os.path.join(out_directory, ytest_file), header=False)

In [ ]:
dfX

In [ ]:
# Plot experimental X predited values of y for calibration

source = ColumnDataSource(
    data=dict(
        y=y[test],
        y_pred=ext.ypred,
    )
)

TOOLTIPS = [("y", "@y"), ("y_pred", "@y_pred")]

output_notebook()
p = figure(
    title="External validation prediction",
    x_axis_label="Experimental pIC50",
    y_axis_label="Predicted pIC50",
    tooltips=TOOLTIPS,
)
# p.text(x+0.3,y+0.3,flav.index[i])
p.circle("y", "y_pred", source=source)
p.line(y[:, 0], y[:, 0], color="red")
export_png(
    p, filename=os.path.join(out_directory, re.sub(r"\D*", "", Xfile) + "_test.png")
)
show(p)

In [ ]:
# Plot experimental X predited values of y for calibration
output_notebook()

source = ColumnDataSource(
    data=dict(
        y_test=y[test],
        y_pred=ext.ypred,
    )
)

source_cal = ColumnDataSource(data=dict(y_train=y[train], y_cal=cv.ycal[:, 1]))

TOOLTIPS = [("y", "@y"), ("y_pred", "@y_pred")]

output_notebook()
p = figure(
    title="External validation prediction",
    x_axis_label="Experimental pIC50",
    y_axis_label="Predicted pIC50",
    tooltips=TOOLTIPS,
)
# p.text(x+0.3,y+0.3,flav.index[i])
# p.output_backend = "svg"
p.line(y[:, 0], y[:, 0], color="blue")
p.circle("y_test", "y_pred", source=source, color="red", legend="Test Set")
p.circle("y_train", "y_cal", source=source_cal, color="black", legend="Training Set")
p.legend.location = "top_left"
export_png(
    p, filename=os.path.join(out_directory, re.sub(r"\D*", "", Xfile) + "_predict.png")
)
show(p)

In [ ]:
ext.searchValidExtVal2(directory, n_test=6, n_splits=500)

In [ ]:
ext.ypred

In [ ]:
ext.y[test]

In [ ]:
calcR2(ext.y[test][:-1], ext.ypred[:-1])